In [5]:
import sys
import time
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import pandas as pd
import numpy as np
import requests

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

print("Food Price Forecasting")
print("FAST data ingestion pipeline initialized.")

# ============================================================
# 1. PROJECT DIRECTORIES
# ============================================================

PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
MODEL_DIR = PROJECT_ROOT / "models"

for directory in [
    DATA_DIR,
    RAW_DIR,
    PROCESSED_DIR,
    OUTPUT_DIR,
    MODEL_DIR
]:
    directory.mkdir(parents=True, exist_ok=True)

print("\nProject directories prepared.")
print("Project root:", PROJECT_ROOT)
print("Raw data directory:", RAW_DIR)
print("Processed data directory:", PROCESSED_DIR)
print("Output directory:", OUTPUT_DIR)
print("Model directory:", MODEL_DIR)

# ============================================================
# 2. FAOSTAT — OPTIONAL / NON-BLOCKING
# ============================================================

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

try:
    from config import FAOSTAT_API_TOKEN
except Exception:
    FAOSTAT_API_TOKEN = ""

FAOSTAT_BASE_URL = "https://faostatservices.fao.org/api/v1/en"

FAOSTAT_DOMAINS = {
    "PP": "Producer Prices",
    "QCL": "Crops and livestock products",
    "QI": "Production Indices",
    "QV": "Value of Agricultural Production",
    "CP": "Consumer Price Indices",
    "TCL": "Crops and livestock products"
}

print("\nFAOSTAT configuration loaded.")
print("Known project domains:", ", ".join(FAOSTAT_DOMAINS.keys()))


def faostat_get(path, params=None, timeout=10):
    url = f"{FAOSTAT_BASE_URL}/{path.lstrip('/')}"

    try:
        response = requests.get(
            url,
            headers={
                "Authorization": f"Bearer {FAOSTAT_API_TOKEN}",
                "Accept": "application/json"
            },
            params=params,
            timeout=timeout
        )

        if response.status_code == 403:
            print(f"\nFAOSTAT 403 skipped: {url}")
            return None

        if not response.ok:
            print(
                f"\nFAOSTAT request skipped: "
                f"HTTP {response.status_code}"
            )
            return None

        try:
            return response.json()
        except ValueError:
            return None

    except requests.RequestException as error:
        print("\nFAOSTAT request skipped:", error)
        return None


producer_prices_dimensions = faostat_get(
    "dimensions/PP"
)

if producer_prices_dimensions:
    print(
        "\nFAOSTAT Producer Prices dimensions available."
    )
else:
    print(
        "\nFAOSTAT Producer Prices dimensions unavailable; "
        "continuing with AGMARKNET."
    )

print(
    "FAOSTAT investigation will not block the project."
)

# ============================================================
# 3. AGMARKNET CONFIGURATION
# ============================================================

AGMARKNET_BASE_URL = (
    "https://api.agmarknet.gov.in/v1"
)

agmarknet_headers = {
    "Accept": "application/json, text/plain, */*",
    "Origin": "https://agmarknet.gov.in",
    "Referer": "https://agmarknet.gov.in/",
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/153.0.0.0 Safari/537.36"
    )
}

FILTERS_URL = (
    f"{AGMARKNET_BASE_URL}/daily-price-arrival/filters"
)

PRICE_URL = (
    f"{AGMARKNET_BASE_URL}/prices-and-arrivals/"
    "market-report/specific"
)

print("\nTesting AGMARKNET connectivity...")

response = requests.get(
    FILTERS_URL,
    headers=agmarknet_headers,
    timeout=12
)

response.raise_for_status()

agmarknet_data = response.json().get(
    "data",
    {}
)

print(
    "AGMARKNET status:",
    response.status_code
)

print(
    "Response type:",
    response.headers.get("content-type")
)

print(
    "AGMARKNET API is reachable."
)

print("\nAGMARKNET filter groups:")

for key, value in agmarknet_data.items():

    if isinstance(value, list):
        print(
            f" - {key}: {len(value)} records"
        )

    elif isinstance(value, dict):
        print(
            f" - {key}: dictionary"
        )

# ============================================================
# 4. COMMODITY INSPECTION
# ============================================================

cmdt_records = agmarknet_data.get(
    "cmdt_data",
    []
)

target_crop_keywords = [
    "wheat",
    "rice",
    "maize",
    "corn",
    "potato",
    "tomato",
    "onion",
    "sugarcane",
    "cotton",
    "groundnut",
    "lentil",
    "pigeon pea",
    "pigeonpea",
    "tur",
    "sorghum",
    "jowar",
    "millet",
    "bajra",
    "barley",
    "banana",
    "sweet potato"
]

commodity_matches = []

for record in cmdt_records:

    commodity_name = str(
        record.get(
            "cmdt_name",
            ""
        )
    )

    if any(
        keyword.lower()
        in commodity_name.lower()
        for keyword in target_crop_keywords
    ):

        commodity_matches.append({
            "Commodity ID":
                record.get("cmdt_id"),

            "Commodity":
                commodity_name,

            "Group ID":
                record.get("cmdt_group_id")
        })


commodity_matches_df = pd.DataFrame(
    commodity_matches
).drop_duplicates()

print(
    "\nMatching AGMARKNET commodities:",
    len(commodity_matches_df)
)

if not commodity_matches_df.empty:
    print(
        commodity_matches_df.to_string(
            index=False
        )
    )

# ============================================================
# 5. VERIFIED PROJECT CROP SET
# ============================================================

final_agmarknet_crops = {

    "Wheat": {
        "commodity_id": 1,
        "group_id": 1
    },

    "Rice": {
        "commodity_id": 3,
        "group_id": 1
    },

    "Maize": {
        "commodity_id": 4,
        "group_id": 1
    },

    "Jowar/Sorghum": {
        "commodity_id": 5,
        "group_id": 1
    },

    "Groundnut": {
        "commodity_id": 10,
        "group_id": 3
    },

    "Cotton": {
        "commodity_id": 15,
        "group_id": 4
    },

    "Banana": {
        "commodity_id": 19,
        "group_id": 5
    },

    "Onion": {
        "commodity_id": 23,
        "group_id": 6
    },

    "Potato": {
        "commodity_id": 24,
        "group_id": 6
    },

    "Lentil": {
        "commodity_id": 52,
        "group_id": 2
    },

    "Sweet Potato": {
        "commodity_id": 124,
        "group_id": 6
    },

    "Bajra/Pearl Millet": {
        "commodity_id": 28,
        "group_id": 1
    }
}

print("\nFinal AGMARKNET crop set:")

for crop_name, info in final_agmarknet_crops.items():

    print(
        f" - {crop_name}: "
        f"commodity={info['commodity_id']}, "
        f"group={info['group_id']}"
    )

# ============================================================
# 6. QUICK ENDPOINT TEST
# ============================================================

test_response = requests.get(
    PRICE_URL,
    headers=agmarknet_headers,
    params={
        "date": "2026-09-01",
        "commodityGroupId": 1,
        "commodityId": 1
    },
    timeout=12
)

print(
    "\nAGMARKNET specific price endpoint status:",
    test_response.status_code
)

test_response.raise_for_status()

test_json = test_response.json()

print(
    "Title:",
    test_json.get("title")
)

print(
    "Commodity:",
    test_json.get("commodityName")
)

print(
    "States:",
    len(
        test_json.get(
            "states",
            []
        )
    )
)

# ============================================================
# 7. FAST INGESTION SETTINGS
# ============================================================

INGESTION_YEARS = range(
    2015,
    2026
)

PRIMARY_DAY = 1

FALLBACK_DAYS = [
    8,
    15
]

REQUEST_TIMEOUT = 10

MAX_RETRIES = 1

MAX_WORKERS = 5

RAW_PRICE_FILE = (
    RAW_DIR /
    "agmarknet_price_observations.csv"
)

LOG_FILE = (
    OUTPUT_DIR /
    "agmarknet_ingestion_log.csv"
)

MONTHLY_PRICE_FILE = (
    PROCESSED_DIR /
    "monthly_crop_prices.csv"
)

print(
    "\nFAST ingestion configuration:"
)

print(
    "Years:",
    f"{min(INGESTION_YEARS)}–"
    f"{max(INGESTION_YEARS)}"
)

print(
    "Crops:",
    len(final_agmarknet_crops)
)

print(
    "Primary day:",
    PRIMARY_DAY
)

print(
    "Fallback days:",
    FALLBACK_DAYS
)

print(
    "Parallel workers:",
    MAX_WORKERS
)

print(
    "Existing raw file:",
    RAW_PRICE_FILE
)

# ============================================================
# 8. LOAD EXISTING PROGRESS
# ============================================================

if RAW_PRICE_FILE.exists():

    try:
        raw_existing_df = pd.read_csv(
            RAW_PRICE_FILE
        )
    except Exception:
        raw_existing_df = pd.DataFrame()

else:
    raw_existing_df = pd.DataFrame()


if LOG_FILE.exists():

    try:
        ingestion_log_df = pd.read_csv(
            LOG_FILE
        )
    except Exception:
        ingestion_log_df = pd.DataFrame()

else:
    ingestion_log_df = pd.DataFrame()


print(
    "\nExisting raw observations:",
    f"{len(raw_existing_df):,}"
)

print(
    "Existing log entries:",
    f"{len(ingestion_log_df):,}"
)

# ============================================================
# 9. DETERMINE ALREADY COMPLETED MONTHS
# ============================================================

completed_month_keys = set()

if not ingestion_log_df.empty:

    required_log_columns = {
        "Crop",
        "Year",
        "Month",
        "Success",
        "Observations"
    }

    if required_log_columns.issubset(
        ingestion_log_df.columns
    ):

        temp_log = ingestion_log_df.copy()

        temp_log["Year_num"] = pd.to_numeric(
            temp_log["Year"],
            errors="coerce"
        )

        temp_log["Month_num"] = pd.to_numeric(
            temp_log["Month"],
            errors="coerce"
        )

        temp_log["Success_bool"] = (
            temp_log["Success"]
            .astype(str)
            .str.lower()
            .isin(
                [
                    "true",
                    "1",
                    "yes"
                ]
            )
        )

        temp_log["Obs_num"] = (
            pd.to_numeric(
                temp_log["Observations"],
                errors="coerce"
            )
            .fillna(0)
        )

        valid_log = temp_log[
            temp_log["Success_bool"]
            &
            (
                temp_log["Obs_num"] > 0
            )
        ]

        for row in (
            valid_log[
                [
                    "Crop",
                    "Year_num",
                    "Month_num"
                ]
            ]
            .drop_duplicates()
            .itertuples(
                index=False
            )
        ):

            completed_month_keys.add(
                (
                    str(row.Crop),
                    int(row.Year_num),
                    int(row.Month_num)
                )
            )


print(
    "Completed crop-months already available:",
    len(completed_month_keys)
)

# ============================================================
# 10. REQUEST / FLATTENING FUNCTION
# ============================================================

def fetch_agmarknet_day(
    crop_name,
    crop_info,
    date_string
):

    params = {
        "date": date_string,
        "commodityGroupId":
            crop_info["group_id"],
        "commodityId":
            crop_info["commodity_id"]
    }

    last_error = ""
    last_status = None

    for attempt in range(
        MAX_RETRIES + 1
    ):

        try:

            response = requests.get(
                PRICE_URL,
                headers=agmarknet_headers,
                params=params,
                timeout=REQUEST_TIMEOUT
            )

            last_status = (
                response.status_code
            )

            if response.ok:

                data = response.json()

                records = []

                for state in (
                    data.get(
                        "states",
                        []
                    )
                ):

                    state_name = (
                        state.get(
                            "stateName"
                        )
                    )

                    state_id = (
                        state.get(
                            "stateId"
                        )
                    )

                    for market in (
                        state.get(
                            "markets",
                            []
                        )
                    ):

                        market_name = (
                            market.get(
                                "marketName"
                            )
                        )

                        market_id = (
                            market.get(
                                "marketId"
                            )
                        )

                        for record in (
                            market.get(
                                "data",
                                []
                            )
                        ):

                            records.append({

                                "Date":
                                    date_string,

                                "Crop":
                                    crop_name,

                                "Commodity ID":
                                    crop_info[
                                        "commodity_id"
                                    ],

                                "Commodity Group ID":
                                    crop_info[
                                        "group_id"
                                    ],

                                "State":
                                    state_name,

                                "State ID":
                                    state_id,

                                "Market":
                                    market_name,

                                "Market ID":
                                    market_id,

                                "Variety":
                                    record.get(
                                        "variety"
                                    ),

                                "Grade":
                                    record.get(
                                        "grade"
                                    ),

                                "Minimum Price":
                                    record.get(
                                        "minimumPrice"
                                    ),

                                "Maximum Price":
                                    record.get(
                                        "maximumPrice"
                                    ),

                                "Modal Price":
                                    record.get(
                                        "modalPrice"
                                    ),

                                "Price Unit":
                                    record.get(
                                        "unitOfPrice"
                                    ),

                                "Arrival":
                                    record.get(
                                        "arrivals"
                                    ),

                                "Arrival Unit":
                                    record.get(
                                        "unitOfArrivals"
                                    )
                            })


                usable_records = [

                    record

                    for record in records

                    if pd.to_numeric(
                        record.get(
                            "Modal Price"
                        ),
                        errors="coerce"
                    ) > 0

                ]

                metadata = {

                    "Crop":
                        crop_name,

                    "Year":
                        int(
                            date_string[:4]
                        ),

                    "Month":
                        int(
                            date_string[5:7]
                        ),

                    "Date":
                        date_string,

                    "HTTP_Status":
                        response.status_code,

                    "Success":
                        (
                            bool(
                                data.get(
                                    "success"
                                )
                            )
                            and
                            len(
                                usable_records
                            ) > 0
                        ),

                    "States":
                        len(
                            data.get(
                                "states",
                                []
                            )
                        ),

                    "Markets":
                        sum(
                            len(
                                state.get(
                                    "markets",
                                    []
                                )
                            )
                            for state in (
                                data.get(
                                    "states",
                                    []
                                )
                            )
                        ),

                    "Observations":
                        len(records),

                    "Error":
                        ""
                }

                return (
                    records,
                    metadata
                )

            last_error = (
                response.text[:300]
            )

        except (
            requests.RequestException,
            ValueError
        ) as error:

            last_error = str(error)

        if attempt < MAX_RETRIES:
            time.sleep(0.35)

    return (
        [],
        {
            "Crop":
                crop_name,

            "Year":
                int(
                    date_string[:4]
                ),

            "Month":
                int(
                    date_string[5:7]
                ),

            "Date":
                date_string,

            "HTTP_Status":
                last_status,

            "Success":
                False,

            "States":
                0,

            "Markets":
                0,

            "Observations":
                0,

            "Error":
                last_error
        }
    )

# ============================================================
# 11. BUILD ONLY MISSING REQUESTS
# ============================================================

tasks = []

for crop_name, crop_info in (
    final_agmarknet_crops.items()
):

    for year in INGESTION_YEARS:

        for month in range(
            1,
            13
        ):

            if (
                crop_name,
                year,
                month
            ) in completed_month_keys:

                continue

            tasks.append(
                (
                    crop_name,
                    crop_info,
                    year,
                    month
                )
            )


total_possible = (
    len(final_agmarknet_crops)
    *
    len(list(INGESTION_YEARS))
    *
    12
)

print(
    "\n" + "=" * 90
)

print(
    "FAST RESUMABLE AGMARKNET INGESTION"
)

print(
    "=" * 90
)

print(
    "Total crop-months in project:",
    total_possible
)

print(
    "Already completed:",
    len(completed_month_keys)
)

print(
    "Crop-months still requiring work:",
    len(tasks)
)

if not tasks:

    print(
        "\nAll crop-months already have "
        "usable observations."
    )

else:

    print(
        "\nMaximum normal requests for this run:",
        len(tasks),
        "(plus fallback requests only when necessary)"
    )

# ============================================================
# 12. FAST MONTH FETCH
# ============================================================

def fetch_month(task):

    crop_name, crop_info, year, month = task

    candidate_days = (
        [PRIMARY_DAY]
        +
        FALLBACK_DAYS
    )

    attempts = []

    for day in candidate_days:

        try:

            date_obj = pd.Timestamp(
                year=year,
                month=month,
                day=day
            )

        except ValueError:

            continue

        date_string = (
            date_obj.strftime(
                "%Y-%m-%d"
            )
        )

        records, metadata = (
            fetch_agmarknet_day(
                crop_name,
                crop_info,
                date_string
            )
        )

        attempts.append(
            (
                records,
                metadata
            )
        )

        if (
            metadata["Success"]
            and records
        ):

            return (
                crop_name,
                year,
                month,
                records,
                metadata,
                attempts
            )

    if attempts:

        records, metadata = attempts[-1]

        return (
            crop_name,
            year,
            month,
            records,
            metadata,
            attempts
        )

    return (
        crop_name,
        year,
        month,
        [],
        {
            "Crop":
                crop_name,

            "Year":
                year,

            "Month":
                month,

            "Date":
                "",

            "HTTP_Status":
                None,

            "Success":
                False,

            "States":
                0,

            "Markets":
                0,

            "Observations":
                0,

            "Error":
                "No valid candidate date"
        },
        []
    )

# ============================================================
# 13. SAVE HELPER
# ============================================================

def append_df(
    df,
    filepath
):

    if (
        df is None
        or df.empty
    ):

        return

    filepath.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    header = (
        not filepath.exists()
    )

    df.to_csv(
        filepath,
        mode="a",
        header=header,
        index=False
    )

# ============================================================
# 14. PARALLEL INGESTION
# ============================================================

new_raw_rows = 0
completed_this_run = 0
failed_this_run = 0
successful_dates_this_run = 0

try:

    with ThreadPoolExecutor(
        max_workers=MAX_WORKERS
    ) as executor:

        future_map = {
            executor.submit(
                fetch_month,
                task
            ): task

            for task in tasks
        }

        for completed_count, future in enumerate(
            as_completed(
                future_map
            ),
            start=1
        ):

            task = future_map[future]

            try:

                (
                    crop_name,
                    year,
                    month,
                    records,
                    metadata,
                    attempts
                ) = future.result()

            except Exception as error:

                crop_name, crop_info, year, month = task

                records = []

                metadata = {

                    "Crop":
                        crop_name,

                    "Year":
                        year,

                    "Month":
                        month,

                    "Date":
                        "",

                    "HTTP_Status":
                        None,

                    "Success":
                        False,

                    "States":
                        0,

                    "Markets":
                        0,

                    "Observations":
                        0,

                    "Error":
                        str(error)

                }

                attempts = []

            for (
                _,
                attempt_metadata
            ) in attempts:

                append_df(
                    pd.DataFrame(
                        [attempt_metadata]
                    ),
                    LOG_FILE
                )

            if (
                metadata.get(
                    "Success"
                )
                and
                records
            ):

                raw_day_df = pd.DataFrame(
                    records
                )

                for column in [
                    "Minimum Price",
                    "Maximum Price",
                    "Modal Price",
                    "Arrival"
                ]:

                    if column in raw_day_df.columns:

                        raw_day_df[column] = (
                            pd.to_numeric(
                                raw_day_df[column],
                                errors="coerce"
                            )
                        )

                raw_day_df = (
                    raw_day_df.drop_duplicates(
                        subset=[
                            "Date",
                            "Crop",
                            "State ID",
                            "Market ID",
                            "Variety",
                            "Grade"
                        ]
                    )
                )

                append_df(
                    raw_day_df,
                    RAW_PRICE_FILE
                )

                new_raw_rows += len(
                    raw_day_df
                )

                completed_this_run += 1

                successful_dates_this_run += 1

            else:

                failed_this_run += 1

            if (
                completed_count % 10 == 0
                or
                completed_count == len(tasks)
            ):

                print(
                    f"Progress: "
                    f"{completed_count:,}/"
                    f"{len(tasks):,} "
                    f"| successful: "
                    f"{completed_this_run:,} "
                    f"| failed: "
                    f"{failed_this_run:,} "
                    f"| new raw rows: "
                    f"{new_raw_rows:,}"
                )

except KeyboardInterrupt:

    print(
        "\nKeyboardInterrupt received."
    )

    print(
        "Completed futures have already been saved."
    )

    print(
        "Rerun this cell to resume."
    )


print(
    "\n" + "=" * 90
)

print(
    "FAST INGESTION STAGE COMPLETE"
)

print(
    "=" * 90
)

print(
    "Successful crop-months this run:",
    completed_this_run
)

print(
    "Failed crop-months this run:",
    failed_this_run
)

print(
    "New raw rows written:",
    f"{new_raw_rows:,}"
)

# ============================================================
# 15. LOAD, CLEAN AND DEDUPLICATE RAW DATA
# ============================================================

if RAW_PRICE_FILE.exists():

    raw_price_df = pd.read_csv(
        RAW_PRICE_FILE
    )

else:

    raw_price_df = pd.DataFrame()


print(
    "\nRaw AGMARKNET observations loaded:",
    f"{len(raw_price_df):,}"
)

if not raw_price_df.empty:

    raw_price_df["Date"] = pd.to_datetime(
        raw_price_df["Date"],
        errors="coerce"
    )

    for column in [
        "Modal Price",
        "Minimum Price",
        "Maximum Price",
        "Arrival"
    ]:

        if column in raw_price_df.columns:

            raw_price_df[column] = (
                pd.to_numeric(
                    raw_price_df[column],
                    errors="coerce"
                )
            )

    raw_price_df = raw_price_df[
        raw_price_df["Date"].notna()
        &
        raw_price_df["Modal Price"].notna()
        &
        (
            raw_price_df["Modal Price"] > 0
        )
    ].copy()

    raw_price_df = (
        raw_price_df.drop_duplicates(
            subset=[
                "Date",
                "Crop",
                "State ID",
                "Market ID",
                "Variety",
                "Grade"
            ]
        )
        .reset_index(drop=True)
    )

    raw_price_df["Year"] = (
        raw_price_df["Date"].dt.year
    )

    raw_price_df["Month"] = (
        raw_price_df["Date"].dt.month
    )

    raw_price_df.to_csv(
        RAW_PRICE_FILE,
        index=False
    )

    print(
        "Usable raw observations:",
        f"{len(raw_price_df):,}"
    )

# ============================================================
# 16. MONTHLY NATIONAL PRICE DATASET
# ============================================================

if raw_price_df.empty:

    monthly_price_df = pd.DataFrame()

else:

    monthly_price_df = (
        raw_price_df
        .groupby(
            [
                "Crop",
                "Year",
                "Month"
            ],
            as_index=False
        )
        .agg(
            Price=(
                "Modal Price",
                "median"
            ),

            Mean_Price=(
                "Modal Price",
                "mean"
            ),

            Min_Price=(
                "Modal Price",
                "min"
            ),

            Max_Price=(
                "Modal Price",
                "max"
            ),

            Price_Observations=(
                "Modal Price",
                "count"
            ),

            Markets=(
                "Market ID",
                "nunique"
            ),

            States=(
                "State ID",
                "nunique"
            )
        )
        .sort_values(
            [
                "Crop",
                "Year",
                "Month"
            ]
        )
        .reset_index(drop=True)
    )

    monthly_price_df["Date"] = pd.to_datetime(
        monthly_price_df[
            [
                "Year",
                "Month"
            ]
        ].assign(
            Day=1
        )
    )

    monthly_price_df = monthly_price_df[
        [
            "Date",
            "Year",
            "Month",
            "Crop",
            "Price",
            "Mean_Price",
            "Min_Price",
            "Max_Price",
            "Price_Observations",
            "Markets",
            "States"
        ]
    ]

    monthly_price_df.to_csv(
        MONTHLY_PRICE_FILE,
        index=False
    )

# ============================================================
# 17. FINAL DATA QUALITY CHECK
# ============================================================

print(
    "\n" + "=" * 90
)

print(
    "FINAL DATA QUALITY CHECK"
)

print(
    "=" * 90
)

if raw_price_df.empty:

    print(
        "No raw observations available."
    )

else:

    duplicate_key_count = (
        raw_price_df.duplicated(
            subset=[
                "Date",
                "Crop",
                "State ID",
                "Market ID",
                "Variety",
                "Grade"
            ]
        ).sum()
    )

    print(
        "Raw observations:",
        f"{len(raw_price_df):,}"
    )

    print(
        "Exact duplicate keys:",
        duplicate_key_count
    )

    print(
        "Unique crops:",
        raw_price_df["Crop"].nunique()
    )

    print(
        "Unique markets:",
        raw_price_df["Market ID"].nunique()
    )

    print(
        "Unique states:",
        raw_price_df["State ID"].nunique()
    )

    print(
        "Date range:",
        raw_price_df["Date"].min().date(),
        "to",
        raw_price_df["Date"].max().date()
    )

if monthly_price_df.empty:

    print(
        "\nMonthly dataset is empty."
    )

else:

    monthly_duplicate_count = (
        monthly_price_df.duplicated(
            subset=[
                "Crop",
                "Year",
                "Month"
            ]
        ).sum()
    )

    print(
        "\nMonthly rows:",
        f"{len(monthly_price_df):,}"
    )

    print(
        "Monthly duplicate "
        "Crop-Year-Month keys:",
        monthly_duplicate_count
    )

    print(
        "Crops represented:",
        monthly_price_df["Crop"].nunique()
    )

    print(
        "Monthly date range:",
        monthly_price_df["Date"].min().date(),
        "to",
        monthly_price_df["Date"].max().date()
    )

    print(
        "\nRows per crop:"
    )

    print(
        monthly_price_df
        .groupby("Crop")
        .size()
        .sort_values(ascending=False)
        .to_string()
    )

    print(
        "\nSample:"
    )

    print(
        monthly_price_df
        .head(20)
        .to_string(index=False)
    )

# ============================================================
# 18. FINAL STATUS
# ============================================================

print(
    "\n" + "=" * 90
)

print(
    "FOOD PRICE FORECASTING — DATA INGESTION STATUS"
)

print(
    "=" * 90
)

print(
    "Primary price source:",
    "Government of India AGMARKNET"
)

print(
    "Forecasting price field:",
    "Modal Price (Rs./Quintal)"
)

print(
    "Raw granularity:",
    "Date + Crop + State + Market + Variety + Grade"
)

print(
    "Monthly target:",
    "National median market Modal Price"
)

print(
    "FAOSTAT:",
    "Optional enrichment source; "
    "403 responses do not block ingestion."
)

print(
    "\nRaw dataset:",
    RAW_PRICE_FILE
)

print(
    "Monthly forecasting dataset:",
    MONTHLY_PRICE_FILE
)

print(
    "\nNext stages:"
    "\n1. Feature engineering"
    "\n2. Exploratory analysis"
    "\n3. Forecasting models"
    "\n4. Model evaluation"
)

Food Price Forecasting
FAST data ingestion pipeline initialized.

Project directories prepared.
Project root: d:\OneDrive\Documents\Desktop\Resume Projects\Food Price Forecasting
Raw data directory: d:\OneDrive\Documents\Desktop\Resume Projects\Food Price Forecasting\data\raw
Processed data directory: d:\OneDrive\Documents\Desktop\Resume Projects\Food Price Forecasting\data\processed
Output directory: d:\OneDrive\Documents\Desktop\Resume Projects\Food Price Forecasting\outputs
Model directory: d:\OneDrive\Documents\Desktop\Resume Projects\Food Price Forecasting\models

FAOSTAT configuration loaded.
Known project domains: PP, QCL, QI, QV, CP, TCL

FAOSTAT 403 skipped: https://faostatservices.fao.org/api/v1/en/dimensions/PP

FAOSTAT Producer Prices dimensions unavailable; continuing with AGMARKNET.
FAOSTAT investigation will not block the project.

Testing AGMARKNET connectivity...
AGMARKNET status: 200
Response type: application/json
AGMARKNET API is reachable.

AGMARKNET filter groups:

ParserError: Error tokenizing data. C error: Expected 14 fields in line 23389, saw 16


In [6]:
from pathlib import Path
import csv
import pandas as pd

RAW_PRICE_FILE = Path(PROJECT_ROOT) / "data" / "raw" / "agmarknet_price_observations.csv"

print("=" * 90)
print("RAW CSV VALIDATION")
print("=" * 90)

print(f"File: {RAW_PRICE_FILE}")
print(f"Exists: {RAW_PRICE_FILE.exists()}")

if not RAW_PRICE_FILE.exists():
    raise FileNotFoundError(f"Raw price file not found: {RAW_PRICE_FILE}")

file_size_mb = RAW_PRICE_FILE.stat().st_size / (1024 * 1024)
print(f"File size: {file_size_mb:.2f} MB")

# ------------------------------------------------------------
# Inspect the CSV structure safely
# ------------------------------------------------------------

with open(
    RAW_PRICE_FILE,
    "r",
    encoding="utf-8-sig",
    newline=""
) as f:
    reader = csv.reader(f)

    header = next(reader)
    expected_fields = len(header)

    print(f"\nExpected columns: {expected_fields}")
    print("Columns:")
    for i, column in enumerate(header, start=1):
        print(f"  {i}. {column}")

    malformed_rows = []
    total_rows = 0

    for line_number, row in enumerate(reader, start=2):
        total_rows += 1

        if len(row) != expected_fields:
            malformed_rows.append(
                {
                    "line_number": line_number,
                    "field_count": len(row),
                    "expected": expected_fields,
                    "row": row
                }
            )

print("\n" + "-" * 90)
print(f"Total data rows scanned: {total_rows:,}")
print(f"Malformed rows found: {len(malformed_rows):,}")

if malformed_rows:
    print("\nFirst malformed rows:")

    for bad in malformed_rows[:10]:
        print(
            f"\nLine {bad['line_number']}: "
            f"found {bad['field_count']} fields, "
            f"expected {bad['expected']} fields"
        )

        print(bad["row"])

else:
    print("\nNo malformed rows found.")

print("\nRaw CSV validation complete.")

RAW CSV VALIDATION
File: d:\OneDrive\Documents\Desktop\Resume Projects\Food Price Forecasting\data\raw\agmarknet_price_observations.csv
Exists: True
File size: 54.24 MB

Expected columns: 14
Columns:
  1. Date
  2. Crop
  3. Commodity ID
  4. Commodity Group ID
  5. State
  6. State ID
  7. Market
  8. Market ID
  9. Variety
  10. Grade
  11. Minimum Price
  12. Maximum Price
  13. Modal Price
  14. Unit

------------------------------------------------------------------------------------------
Total data rows scanned: 473,488
Malformed rows found: 450,101

First malformed rows:

Line 23389: found 16 fields, expected 14 fields
['2015-01-01', 'Wheat', '1', '1', 'Andhra Pradesh', '2', 'Visakhapatnam APMC', '838', 'Lok-1', 'Medium', '2100.0', '2300.0', '2200.0', 'Rs./Quintal', '0.1', 'Metric Tonnes']

Line 23390: found 16 fields, expected 14 fields
['2015-01-01', 'Wheat', '1', '1', 'Assam', '4', 'Baharihat APMC', '385', 'Other', 'FAQ', '1500.0', '1550.0', '1525.0', 'Rs./Quintal', '1.0', '

In [7]:
from pathlib import Path
import csv
import pandas as pd

RAW_PRICE_FILE = Path(PROJECT_ROOT) / "data" / "raw" / "agmarknet_price_observations.csv"
CLEAN_RAW_FILE = Path(PROJECT_ROOT) / "data" / "raw" / "agmarknet_price_observations_clean.csv"

print("=" * 90)
print("NORMALIZING AGMARKNET RAW PRICE DATA")
print("=" * 90)

expected_columns = [
    "Date",
    "Crop",
    "Commodity ID",
    "Commodity Group ID",
    "State",
    "State ID",
    "Market",
    "Market ID",
    "Variety",
    "Grade",
    "Minimum Price",
    "Maximum Price",
    "Modal Price",
    "Unit",
]

rows_written = 0
rows_14 = 0
rows_16 = 0
rows_other = 0

with open(
    RAW_PRICE_FILE,
    "r",
    encoding="utf-8-sig",
    newline=""
) as source, open(
    CLEAN_RAW_FILE,
    "w",
    encoding="utf-8",
    newline=""
) as target:

    reader = csv.reader(source)
    writer = csv.writer(target)

    header = next(reader)

    writer.writerow(expected_columns)

    for line_number, row in enumerate(reader, start=2):

        if len(row) == 14:
            clean_row = row
            rows_14 += 1

        elif len(row) == 16:
            # Keep the 14 price-related fields.
            # Fields 15 and 16 are arrivals and arrivals unit.
            clean_row = row[:14]
            rows_16 += 1

        else:
            rows_other += 1
            continue

        writer.writerow(clean_row)
        rows_written += 1

print("\nNormalization complete.")
print("-" * 90)
print(f"Original 14-field rows : {rows_14:,}")
print(f"16-field rows          : {rows_16:,}")
print(f"Other/malformed rows   : {rows_other:,}")
print(f"Clean rows written     : {rows_written:,}")

clean_size_mb = CLEAN_RAW_FILE.stat().st_size / (1024 * 1024)

print(f"\nClean file:")
print(CLEAN_RAW_FILE)
print(f"Clean file size: {clean_size_mb:.2f} MB")

print("\nPrice columns preserved:")
for column in expected_columns:
    print(f" - {column}")

print("\nThe original raw CSV has NOT been modified.")
print("The normalized file is ready for validation.")

NORMALIZING AGMARKNET RAW PRICE DATA

Normalization complete.
------------------------------------------------------------------------------------------
Original 14-field rows : 23,387
16-field rows          : 450,101
Other/malformed rows   : 0
Clean rows written     : 473,488

Clean file:
d:\OneDrive\Documents\Desktop\Resume Projects\Food Price Forecasting\data\raw\agmarknet_price_observations_clean.csv
Clean file size: 46.11 MB

Price columns preserved:
 - Date
 - Crop
 - Commodity ID
 - Commodity Group ID
 - State
 - State ID
 - Market
 - Market ID
 - Variety
 - Grade
 - Minimum Price
 - Maximum Price
 - Modal Price
 - Unit

The original raw CSV has NOT been modified.
The normalized file is ready for validation.


In [8]:
import pandas as pd
from pathlib import Path

CLEAN_RAW_FILE = Path(PROJECT_ROOT) / "data" / "raw" / "agmarknet_price_observations_clean.csv"

print("=" * 90)
print("VALIDATING CLEAN AGMARKNET PRICE DATA")
print("=" * 90)

df_clean = pd.read_csv(CLEAN_RAW_FILE)

print(f"\nRows: {len(df_clean):,}")
print(f"Columns: {len(df_clean.columns)}")

print("\nColumn names:")
for column in df_clean.columns:
    print(f" - {column}")

print("\nData types:")
print(df_clean.dtypes)

print("\nMissing values:")
missing = df_clean.isna().sum()
print(missing[missing > 0])

print("\nUnique crops:")
for crop in sorted(df_clean["Crop"].dropna().unique()):
    count = (df_clean["Crop"] == crop).sum()
    print(f" - {crop}: {count:,}")

print("\nDate coverage:")
df_clean["Date"] = pd.to_datetime(df_clean["Date"], errors="coerce")

print(f"Minimum date: {df_clean['Date'].min()}")
print(f"Maximum date: {df_clean['Date'].max()}")

print("\nPrice validation:")

for column in ["Minimum Price", "Maximum Price", "Modal Price"]:
    df_clean[column] = pd.to_numeric(
        df_clean[column],
        errors="coerce"
    )

    print(
        f"{column}: "
        f"missing={df_clean[column].isna().sum():,}, "
        f"negative={(df_clean[column] < 0).sum():,}, "
        f"zero={(df_clean[column] == 0).sum():,}"
    )

print("\nModal price summary:")
print(df_clean["Modal Price"].describe())

print("\nDuplicate check:")

duplicate_columns = [
    "Date",
    "Crop",
    "State ID",
    "Market ID",
    "Variety",
    "Grade"
]

duplicate_count = df_clean.duplicated(
    subset=duplicate_columns
).sum()

print(
    f"Duplicate observations using "
    f"Date + Crop + State + Market + Variety + Grade: "
    f"{duplicate_count:,}"
)

print("\nValidation complete.")

VALIDATING CLEAN AGMARKNET PRICE DATA

Rows: 473,488
Columns: 14

Column names:
 - Date
 - Crop
 - Commodity ID
 - Commodity Group ID
 - State
 - State ID
 - Market
 - Market ID
 - Variety
 - Grade
 - Minimum Price
 - Maximum Price
 - Modal Price
 - Unit

Data types:
Date                      str
Crop                      str
Commodity ID            int64
Commodity Group ID      int64
State                     str
State ID                int64
Market                    str
Market ID               int64
Variety                   str
Grade                     str
Minimum Price         float64
Maximum Price         float64
Modal Price           float64
Unit                      str
dtype: object

Missing values:
Series([], dtype: int64)

Unique crops:
 - Banana: 73,554
 - Cotton: 24,480
 - Groundnut: 18,851
 - Jowar/Sorghum: 17,692
 - Maize: 45,753
 - Onion: 100,873
 - Potato: 26,927
 - Rice: 55,180
 - Wheat: 110,178

Date coverage:
Minimum date: 2015-01-01 00:00:00
Maximum date: 2025-12-

In [9]:
import pandas as pd
from pathlib import Path

CLEAN_RAW_FILE = Path(PROJECT_ROOT) / "data" / "raw" / "agmarknet_price_observations_clean.csv"

df_clean = pd.read_csv(CLEAN_RAW_FILE)

print("=" * 90)
print("INVESTIGATING DUPLICATE AGMARKNET OBSERVATIONS")
print("=" * 90)

# ------------------------------------------------------------
# Duplicate key
# ------------------------------------------------------------

duplicate_columns = [
    "Date",
    "Crop",
    "State ID",
    "Market ID",
    "Variety",
    "Grade"
]

duplicate_mask = df_clean.duplicated(
    subset=duplicate_columns,
    keep=False
)

duplicates = df_clean.loc[
    duplicate_mask
].copy()

print(f"\nTotal rows: {len(df_clean):,}")
print(f"Rows belonging to duplicate groups: {len(duplicates):,}")

# ------------------------------------------------------------
# Number of duplicate groups
# ------------------------------------------------------------

duplicate_groups = (
    duplicates
    .groupby(duplicate_columns, dropna=False)
    .size()
    .reset_index(name="count")
)

print(f"Duplicate groups: {len(duplicate_groups):,}")

print("\nDuplicate group size distribution:")
print(
    duplicate_groups["count"]
    .value_counts()
    .sort_index()
)

# ------------------------------------------------------------
# Duplicates by crop
# ------------------------------------------------------------

print("\nDuplicate rows by crop:")
print(
    duplicates["Crop"]
    .value_counts()
)

# ------------------------------------------------------------
# Show examples
# ------------------------------------------------------------

print("\n" + "-" * 90)
print("EXAMPLES OF DUPLICATE GROUPS")
print("-" * 90)

for _, group in duplicate_groups.head(10).iterrows():

    mask = pd.Series(True, index=df_clean.index)

    for column in duplicate_columns:
        mask &= df_clean[column].astype(str).eq(
            str(group[column])
        )

    example = df_clean.loc[
        mask,
        duplicate_columns + [
            "Minimum Price",
            "Maximum Price",
            "Modal Price",
            "Unit"
        ]
    ]

    print("\nDuplicate group:")
    print(example.to_string(index=False))

print("\nDuplicate investigation complete.")

INVESTIGATING DUPLICATE AGMARKNET OBSERVATIONS

Total rows: 473,488
Rows belonging to duplicate groups: 46,774
Duplicate groups: 23,387

Duplicate group size distribution:
count
2    23387
Name: count, dtype: int64

Duplicate rows by crop:
Crop
Wheat    46774
Name: count, dtype: int64

------------------------------------------------------------------------------------------
EXAMPLES OF DUPLICATE GROUPS
------------------------------------------------------------------------------------------

Duplicate group:
      Date  Crop  State ID  Market ID Variety  Grade  Minimum Price  Maximum Price  Modal Price        Unit
2015-01-01 Wheat         2        838   Lok-1 Medium         2100.0         2300.0       2200.0 Rs./Quintal
2015-01-01 Wheat         2        838   Lok-1 Medium         2100.0         2300.0       2200.0 Rs./Quintal

Duplicate group:
      Date  Crop  State ID  Market ID Variety Grade  Minimum Price  Maximum Price  Modal Price        Unit
2015-01-01 Wheat         4        3

In [10]:
import pandas as pd
from pathlib import Path

CLEAN_RAW_FILE = (
    Path(PROJECT_ROOT)
    / "data"
    / "raw"
    / "agmarknet_price_observations_clean.csv"
)

MASTER_RAW_FILE = (
    Path(PROJECT_ROOT)
    / "data"
    / "raw"
    / "agmarknet_price_observations_master.csv"
)

print("=" * 90)
print("CREATING MASTER AGMARKNET PRICE DATASET")
print("=" * 90)

df = pd.read_csv(CLEAN_RAW_FILE)

print(f"\nRows before deduplication: {len(df):,}")

duplicate_columns = [
    "Date",
    "Crop",
    "State ID",
    "Market ID",
    "Variety",
    "Grade"
]

# Remove exact duplicate observations while keeping the first copy.
df_master = df.drop_duplicates(
    subset=duplicate_columns,
    keep="first"
).copy()

duplicates_removed = len(df) - len(df_master)

print(f"Duplicate rows removed: {duplicates_removed:,}")
print(f"Rows after deduplication: {len(df_master):,}")

# ------------------------------------------------------------
# Save master dataset
# ------------------------------------------------------------

df_master.to_csv(
    MASTER_RAW_FILE,
    index=False
)

print(f"\nMaster dataset saved to:")
print(MASTER_RAW_FILE)

# ------------------------------------------------------------
# Final duplicate verification
# ------------------------------------------------------------

remaining_duplicates = df_master.duplicated(
    subset=duplicate_columns
).sum()

print("\nFinal duplicate check:")
print(f"Remaining duplicate observations: {remaining_duplicates:,}")

# ------------------------------------------------------------
# Crop coverage
# ------------------------------------------------------------

print("\nCrop coverage:")
crop_summary = (
    df_master
    .groupby("Crop")
    .agg(
        observations=("Crop", "size"),
        first_date=("Date", "min"),
        last_date=("Date", "max"),
        markets=("Market ID", "nunique"),
        states=("State ID", "nunique")
    )
    .sort_values("observations", ascending=False)
)

print(crop_summary.to_string())

print("\nMaster dataset creation complete.")

CREATING MASTER AGMARKNET PRICE DATASET

Rows before deduplication: 473,488
Duplicate rows removed: 23,387
Rows after deduplication: 450,101

Master dataset saved to:
d:\OneDrive\Documents\Desktop\Resume Projects\Food Price Forecasting\data\raw\agmarknet_price_observations_master.csv

Final duplicate check:
Remaining duplicate observations: 0

Crop coverage:
               observations  first_date   last_date  markets  states
Crop                                                                
Onion                100873  2015-01-01  2025-12-01     1598      31
Wheat                 86791  2015-01-01  2025-12-08     1531      19
Banana                73554  2015-01-01  2025-12-08     1105      29
Rice                  55180  2015-01-01  2025-12-08      614      24
Maize                 45753  2015-01-01  2025-12-08     1235      23
Potato                26927  2015-01-01  2020-02-01     1024      30
Cotton                24480  2015-01-01  2025-12-08      671      14
Groundnut         

In [11]:
import pandas as pd
from pathlib import Path

MASTER_RAW_FILE = (
    Path(PROJECT_ROOT)
    / "data"
    / "raw"
    / "agmarknet_price_observations_master.csv"
)

MONTHLY_PRICE_FILE = (
    Path(PROJECT_ROOT)
    / "data"
    / "processed"
    / "monthly_crop_prices.csv"
)

print("=" * 90)
print("BUILDING MONTHLY CROP PRICE DATASET")
print("=" * 90)

# ------------------------------------------------------------
# Load master dataset
# ------------------------------------------------------------

df = pd.read_csv(MASTER_RAW_FILE)

print(f"\nMaster observations loaded: {len(df):,}")

# ------------------------------------------------------------
# Prepare dates and prices
# ------------------------------------------------------------

df["Date"] = pd.to_datetime(
    df["Date"],
    errors="coerce"
)

df["Modal Price"] = pd.to_numeric(
    df["Modal Price"],
    errors="coerce"
)

# Remove rows where the date or modal price cannot be used.
df = df.dropna(
    subset=["Date", "Modal Price"]
).copy()

# ------------------------------------------------------------
# Remove zero modal prices
# ------------------------------------------------------------

zero_prices = (df["Modal Price"] <= 0).sum()

print(f"Non-positive modal prices removed: {zero_prices:,}")

df = df[
    df["Modal Price"] > 0
].copy()

# ------------------------------------------------------------
# Create monthly period
# ------------------------------------------------------------

df["Year"] = df["Date"].dt.year
df["Month"] = df["Date"].dt.month
df["Month Start"] = df["Date"].dt.to_period("M").dt.to_timestamp()

# ------------------------------------------------------------
# Aggregate to national monthly level
# ------------------------------------------------------------

monthly = (
    df
    .groupby(
        ["Crop", "Year", "Month", "Month Start"],
        as_index=False
    )
    .agg(
        Median_Price=("Modal Price", "median"),
        Mean_Price=("Modal Price", "mean"),
        Min_Price=("Modal Price", "min"),
        Max_Price=("Modal Price", "max"),
        Price_Observations=("Modal Price", "size"),
        Markets=("Market ID", "nunique"),
        States=("State ID", "nunique")
    )
)

# ------------------------------------------------------------
# Sort
# ------------------------------------------------------------

monthly = monthly.sort_values(
    ["Crop", "Month Start"]
).reset_index(drop=True)

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

monthly.to_csv(
    MONTHLY_PRICE_FILE,
    index=False
)

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

print(f"\nMonthly rows created: {len(monthly):,}")

print(f"\nSaved to:")
print(MONTHLY_PRICE_FILE)

print("\nMonthly dataset columns:")
for column in monthly.columns:
    print(f" - {column}")

print("\nMonthly coverage by crop:")

coverage = (
    monthly
    .groupby("Crop")
    .agg(
        months=("Month Start", "size"),
        first_month=("Month Start", "min"),
        last_month=("Month Start", "max"),
        total_observations=("Price_Observations", "sum")
    )
    .sort_values("total_observations", ascending=False)
)

print(coverage.to_string())

print("\nFirst 20 monthly records:")
print(
    monthly.head(20).to_string(index=False)
)

print("\nMonthly crop price dataset created successfully.")

BUILDING MONTHLY CROP PRICE DATASET

Master observations loaded: 450,101
Non-positive modal prices removed: 93

Monthly rows created: 1,118

Saved to:
d:\OneDrive\Documents\Desktop\Resume Projects\Food Price Forecasting\data\processed\monthly_crop_prices.csv

Monthly dataset columns:
 - Crop
 - Year
 - Month
 - Month Start
 - Median_Price
 - Mean_Price
 - Min_Price
 - Max_Price
 - Price_Observations
 - Markets
 - States

Monthly coverage by crop:
               months first_month last_month  total_observations
Crop                                                            
Onion             132  2015-01-01 2025-12-01              100862
Wheat             132  2015-01-01 2025-12-01               86776
Banana            132  2015-01-01 2025-12-01               73535
Rice              132  2015-01-01 2025-12-01               55179
Maize             132  2015-01-01 2025-12-01               45742
Potato             62  2015-01-01 2020-02-01               26916
Cotton            132  2015-0

In [12]:
import pandas as pd
from pathlib import Path

MONTHLY_PRICE_FILE = (
    Path(PROJECT_ROOT)
    / "data"
    / "processed"
    / "monthly_crop_prices.csv"
)

monthly = pd.read_csv(MONTHLY_PRICE_FILE)

monthly["Month Start"] = pd.to_datetime(
    monthly["Month Start"]
)

print("=" * 90)
print("MONTHLY COVERAGE GAP ANALYSIS")
print("=" * 90)

# ------------------------------------------------------------
# Intended 12-crop project set
# ------------------------------------------------------------

target_crops = [
    "Wheat",
    "Rice",
    "Maize",
    "Jowar/Sorghum",
    "Groundnut",
    "Cotton",
    "Banana",
    "Onion",
    "Potato",
    "Lentil",
    "Sweet Potato",
    "Bajra/Pearl Millet",
]

# ------------------------------------------------------------
# Expected monthly calendar
# ------------------------------------------------------------

expected_months = pd.date_range(
    start="2015-01-01",
    end="2025-12-01",
    freq="MS"
)

print(f"\nExpected months per crop: {len(expected_months)}")
print(f"Expected total crop-months: {len(target_crops) * len(expected_months):,}")

# ------------------------------------------------------------
# Coverage summary
# ------------------------------------------------------------

print("\nCoverage summary:")

coverage_rows = []

for crop in target_crops:

    actual = set(
        monthly.loc[
            monthly["Crop"] == crop,
            "Month Start"
        ]
    )

    expected = set(expected_months)

    missing = sorted(expected - actual)

    coverage_rows.append(
        {
            "Crop": crop,
            "Available Months": len(actual),
            "Missing Months": len(missing),
            "Coverage %": round(
                len(actual) / len(expected) * 100,
                2
            ),
            "First Available": min(actual) if actual else None,
            "Last Available": max(actual) if actual else None,
        }
    )

coverage = pd.DataFrame(coverage_rows)

print(
    coverage.to_string(index=False)
)

# ------------------------------------------------------------
# Detailed missing month list
# ------------------------------------------------------------

print("\n" + "-" * 90)
print("MISSING CROP-MONTHS")
print("-" * 90)

for crop in target_crops:

    actual = set(
        monthly.loc[
            monthly["Crop"] == crop,
            "Month Start"
        ]
    )

    missing = sorted(
        set(expected_months) - actual
    )

    if missing:

        print(f"\n{crop}: {len(missing)} missing")

        # Group missing months by year
        missing_by_year = {}

        for date in missing:
            year = date.year
            missing_by_year.setdefault(year, []).append(date.month)

        for year, months in missing_by_year.items():
            month_names = ", ".join(
                pd.Timestamp(
                    year=year,
                    month=month,
                    day=1
                ).strftime("%b")
                for month in months
            )

            print(f"  {year}: {month_names}")

    else:
        print(f"\n{crop}: COMPLETE")

# ------------------------------------------------------------
# Current dataset crop list
# ------------------------------------------------------------

print("\n" + "-" * 90)
print("CROPS CURRENTLY PRESENT IN MONTHLY DATA")
print("-" * 90)

for crop in sorted(monthly["Crop"].unique()):
    print(f" - {crop}")

print("\nCoverage gap analysis complete.")

MONTHLY COVERAGE GAP ANALYSIS

Expected months per crop: 132
Expected total crop-months: 1,584

Coverage summary:
              Crop  Available Months  Missing Months  Coverage % First Available Last Available
             Wheat               132               0      100.00      2015-01-01     2025-12-01
              Rice               132               0      100.00      2015-01-01     2025-12-01
             Maize               132               0      100.00      2015-01-01     2025-12-01
     Jowar/Sorghum               132               0      100.00      2015-01-01     2025-12-01
         Groundnut               132               0      100.00      2015-01-01     2025-12-01
            Cotton               132               0      100.00      2015-01-01     2025-12-01
            Banana               132               0      100.00      2015-01-01     2025-12-01
             Onion               132               0      100.00      2015-01-01     2025-12-01
            Potato    

In [13]:
import requests
import time

print("=" * 90)
print("TARGETED AGMARKNET AVAILABILITY CHECK")
print("=" * 90)

AGMARKNET_URL = "https://api.agmarknet.gov.in/v1/prices-and-arrivals/market-report/specific"

# Verified commodity mappings from our earlier investigation
test_crops = {
    "Potato": (24, 6),
    "Lentil": (52, 2),
    "Sweet Potato": (124, 6),
    "Bajra/Pearl Millet": (28, 1),
}

# One representative date per year.
test_dates = [
    "2015-01-01",
    "2017-01-01",
    "2019-01-01",
    "2021-01-01",
    "2023-01-01",
    "2025-01-01",
]

results = []

for crop, (commodity_id, group_id) in test_crops.items():

    print(f"\nTesting: {crop}")

    for date in test_dates:

        params = {
            "date": date,
            "commodityGroupId": group_id,
            "commodityId": commodity_id,
        }

        try:
            response = requests.get(
                AGMARKNET_URL,
                params=params,
                headers=agmarknet_headers,
                timeout=20
            )

            if response.status_code != 200:
                print(
                    f"  {date}: HTTP {response.status_code}"
                )

                results.append({
                    "Crop": crop,
                    "Date": date,
                    "Status": response.status_code,
                    "States": 0,
                    "Observations": 0
                })

                time.sleep(2)
                continue

            payload = response.json()

            states = payload.get("data", {}).get("states", [])

            observation_count = sum(
                len(state.get("data", []))
                for state in states
            )

            print(
                f"  {date}: "
                f"{len(states)} states, "
                f"{observation_count:,} observations"
            )

            results.append({
                "Crop": crop,
                "Date": date,
                "Status": response.status_code,
                "States": len(states),
                "Observations": observation_count
            })

        except Exception as e:

            print(
                f"  {date}: ERROR - {type(e).__name__}"
            )

            results.append({
                "Crop": crop,
                "Date": date,
                "Status": "ERROR",
                "States": 0,
                "Observations": 0
            })

        time.sleep(2)

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

availability_test = pd.DataFrame(results)

print("\n" + "=" * 90)
print("AVAILABILITY TEST SUMMARY")
print("=" * 90)

print(
    availability_test.to_string(index=False)
)

print("\nSuccessful test dates by crop:")

for crop in test_crops:

    crop_results = availability_test[
        availability_test["Crop"] == crop
    ]

    successful = crop_results[
        (crop_results["Status"] == 200)
        & (crop_results["Observations"] > 0)
    ]

    print(
        f" - {crop}: "
        f"{len(successful)}/{len(test_dates)}"
    )

print("\nTargeted availability test complete.")

TARGETED AGMARKNET AVAILABILITY CHECK

Testing: Potato
  2015-01-01: HTTP 429
  2017-01-01: HTTP 429
  2019-01-01: HTTP 429
  2021-01-01: HTTP 429
  2023-01-01: HTTP 429
  2025-01-01: HTTP 429

Testing: Lentil
  2015-01-01: HTTP 429
  2017-01-01: HTTP 429
  2019-01-01: HTTP 429
  2021-01-01: HTTP 429
  2023-01-01: HTTP 429
  2025-01-01: HTTP 429

Testing: Sweet Potato
  2015-01-01: HTTP 429
  2017-01-01: HTTP 429
  2019-01-01: HTTP 429
  2021-01-01: HTTP 429
  2023-01-01: HTTP 429
  2025-01-01: HTTP 429

Testing: Bajra/Pearl Millet
  2015-01-01: HTTP 429
  2017-01-01: HTTP 429
  2019-01-01: HTTP 429
  2021-01-01: HTTP 429
  2023-01-01: HTTP 429
  2025-01-01: HTTP 429

AVAILABILITY TEST SUMMARY
              Crop       Date  Status  States  Observations
            Potato 2015-01-01     429       0             0
            Potato 2017-01-01     429       0             0
            Potato 2019-01-01     429       0             0
            Potato 2021-01-01     429       0            

In [16]:
from pathlib import Path
import pandas as pd

processed_path = Path("../data/processed/monthly_crop_prices.csv")

print("=" * 90)
print("PROCESSED FILE HEADER CHECK")
print("=" * 90)

print(f"\nFile exists: {processed_path.exists()}")
print(f"File path: {processed_path.resolve()}")

# Read only the header
header = pd.read_csv(processed_path, nrows=0)

print("\nColumns found:")
for i, column in enumerate(header.columns, start=1):
    print(f"{i}. {repr(column)}")

print("\nRaw first 3 lines of the file:")
with open(processed_path, "r", encoding="utf-8-sig") as f:
    for _ in range(3):
        line = f.readline()
        if not line:
            break
        print(repr(line[:500]))

PROCESSED FILE HEADER CHECK

File exists: True
File path: D:\OneDrive\Documents\Desktop\Resume Projects\Food Price Forecasting\data\processed\monthly_crop_prices.csv

Columns found:
1. 'Crop'
2. 'Year'
3. 'Month'
4. 'Month Start'
5. 'Median_Price'
6. 'Mean_Price'
7. 'Min_Price'
8. 'Max_Price'
9. 'Price_Observations'
10. 'Markets'
11. 'States'

Raw first 3 lines of the file:
'Crop,Year,Month,Month Start,Median_Price,Mean_Price,Min_Price,Max_Price,Price_Observations,Markets,States\n'
'Banana,2015,1,2015-01-01,2050.0,2225.4681528662422,190.0,5200.0,471,215,20\n'
'Banana,2015,2,2015-02-01,2000.0,1982.352,575.0,4000.0,125,75,13\n'


In [17]:
from pathlib import Path
import pandas as pd

processed_path = Path("../data/processed/monthly_crop_prices.csv")

df_monthly = pd.read_csv(
    processed_path,
    parse_dates=["Month Start"]
)

print("=" * 90)
print("CURRENT MONTHLY DATASET CHECK")
print("=" * 90)

print(f"\nRows: {len(df_monthly):,}")
print(f"Columns: {len(df_monthly.columns)}")

print("\nCrops:")
print(sorted(df_monthly["Crop"].unique()))

print("\nCoverage by crop:")

coverage = (
    df_monthly.groupby("Crop")
    .agg(
        Start_Date=("Month Start", "min"),
        End_Date=("Month Start", "max"),
        Months=("Month Start", "nunique"),
        Observations=("Price_Observations", "sum"),
        Markets=("Markets", "sum"),
        States=("States", "max")
    )
    .sort_index()
)

print(coverage.to_string())

print("\nMissing months within each crop's own date range:")

for crop, group in df_monthly.groupby("Crop"):
    expected_dates = pd.date_range(
        group["Month Start"].min(),
        group["Month Start"].max(),
        freq="MS"
    )

    actual_dates = pd.DatetimeIndex(group["Month Start"])

    missing = expected_dates.difference(actual_dates)

    print(f" - {crop}: {len(missing)} missing months")

print("\nFirst 10 rows:")
display(df_monthly.head(10))

CURRENT MONTHLY DATASET CHECK

Rows: 1,118
Columns: 11

Crops:
['Banana', 'Cotton', 'Groundnut', 'Jowar/Sorghum', 'Maize', 'Onion', 'Potato', 'Rice', 'Wheat']

Coverage by crop:
              Start_Date   End_Date  Months  Observations  Markets  States
Crop                                                                      
Banana        2015-01-01 2025-12-01     132         73535    34155      24
Cotton        2015-01-01 2025-12-01     132         24472    15176      13
Groundnut     2015-01-01 2025-12-01     132         18837    11803      15
Jowar/Sorghum 2015-01-01 2025-12-01     132         17689    11973      11
Maize         2015-01-01 2025-12-01     132         45742    30325      19
Onion         2015-01-01 2025-12-01     132        100862    67648      26
Potato        2015-01-01 2020-02-01      62         26916    26292      27
Rice          2015-01-01 2025-12-01     132         55179    27064      18
Wheat         2015-01-01 2025-12-01     132         86776    59597      

,Crop,Year,Month,Month Start,Median_Price,Mean_Price,Min_Price,Max_Price,Price_Observations,Markets,States
0,Banana,2015,1,2015-01-01,2050.0,2225.468153,190.0,5200.0,471,215,20
1,Banana,2015,2,2015-02-01,2000.0,1982.352000,575.0,4000.0,125,75,13
2,Banana,2015,3,2015-03-01,1960.0,1901.767606,275.0,3600.0,142,93,15
3,Banana,2015,4,2015-04-01,1910.0,1954.390300,85.0,5000.0,433,230,22
4,Banana,2015,5,2015-05-01,1900.0,1932.347619,90.0,6000.0,420,224,21
5,Banana,2015,6,2015-06-01,1900.0,2025.617476,95.0,6300.0,515,229,22
6,Banana,2015,7,2015-07-01,1900.0,2076.219885,65.0,5300.0,523,246,21
7,Banana,2015,8,2015-08-01,1800.0,1987.990450,90.0,5100.0,489,235,22
8,Banana,2015,9,2015-09-01,1900.0,2031.226345,80.0,6000.0,539,243,21
9,Banana,2015,10,2015-10-01,1800.0,2004.604520,85.0,5200.0,531,236,24


In [18]:
print("=" * 90)
print("FINAL PROCESSED DATA QUALITY CHECK")
print("=" * 90)

# ---------------------------------------------------------
# 1. Missing values
# ---------------------------------------------------------
print("\n1. MISSING VALUES")
missing = df_monthly.isna().sum()

if missing.sum() == 0:
    print("✓ No missing values found.")
else:
    print(missing[missing > 0].to_string())


# ---------------------------------------------------------
# 2. Duplicate Crop-Month records
# ---------------------------------------------------------
print("\n2. DUPLICATE CROP-MONTH RECORDS")

duplicates = df_monthly.duplicated(
    subset=["Crop", "Year", "Month"],
    keep=False
)

duplicate_count = duplicates.sum()

if duplicate_count == 0:
    print("✓ No duplicate Crop-Year-Month records found.")
else:
    print(f"⚠ {duplicate_count:,} duplicate rows found.")
    display(
        df_monthly.loc[duplicates]
        .sort_values(["Crop", "Year", "Month"])
        .head(20)
    )


# ---------------------------------------------------------
# 3. Non-positive prices
# ---------------------------------------------------------
print("\n3. PRICE VALIDATION")

price_columns = [
    "Median_Price",
    "Mean_Price",
    "Min_Price",
    "Max_Price"
]

for column in price_columns:
    count = (df_monthly[column] <= 0).sum()
    print(f" - {column}: {count:,} non-positive values")


# ---------------------------------------------------------
# 4. Logical price consistency
# ---------------------------------------------------------
print("\n4. PRICE CONSISTENCY CHECK")

invalid_min_max = (
    df_monthly["Min_Price"] > df_monthly["Max_Price"]
).sum()

invalid_median = (
    (df_monthly["Median_Price"] < df_monthly["Min_Price"]) |
    (df_monthly["Median_Price"] > df_monthly["Max_Price"])
).sum()

invalid_mean = (
    (df_monthly["Mean_Price"] < df_monthly["Min_Price"]) |
    (df_monthly["Mean_Price"] > df_monthly["Max_Price"])
).sum()

print(f" - Min > Max: {invalid_min_max:,}")
print(f" - Median outside Min-Max: {invalid_median:,}")
print(f" - Mean outside Min-Max: {invalid_mean:,}")


# ---------------------------------------------------------
# 5. Observation counts
# ---------------------------------------------------------
print("\n5. OBSERVATION COUNTS")

zero_observations = (
    df_monthly["Price_Observations"] <= 0
).sum()

print(f" - Months with zero price observations: {zero_observations:,}")

print("\nObservation statistics:")
print(
    df_monthly["Price_Observations"]
    .describe()
    .round(2)
    .to_string()
)


# ---------------------------------------------------------
# 6. Overall price statistics by crop
# ---------------------------------------------------------
print("\n6. PRICE STATISTICS BY CROP")

crop_stats = (
    df_monthly.groupby("Crop")["Median_Price"]
    .agg(
        Mean="mean",
        Median="median",
        Minimum="min",
        Maximum="max",
        Std_Dev="std"
    )
    .round(2)
)

print(crop_stats.to_string())


print("\n" + "=" * 90)
print("DATA QUALITY CHECK COMPLETE")
print("=" * 90)

FINAL PROCESSED DATA QUALITY CHECK

1. MISSING VALUES
✓ No missing values found.

2. DUPLICATE CROP-MONTH RECORDS
✓ No duplicate Crop-Year-Month records found.

3. PRICE VALIDATION
 - Median_Price: 0 non-positive values
 - Mean_Price: 0 non-positive values
 - Min_Price: 0 non-positive values
 - Max_Price: 0 non-positive values

4. PRICE CONSISTENCY CHECK
 - Min > Max: 0
 - Median outside Min-Max: 0
 - Mean outside Min-Max: 0

5. OBSERVATION COUNTS
 - Months with zero price observations: 0

Observation statistics:
count    1118.00
mean      402.51
std       289.08
min         6.00
25%       165.00
50%       371.00
75%       532.00
max      1698.00

6. PRICE STATISTICS BY CROP
                  Mean   Median  Minimum  Maximum  Std_Dev
Crop                                                      
Banana         2465.66  2300.00   1500.0   4000.0   599.40
Cotton         5982.73  5491.00   3900.0  10500.0  1532.46
Groundnut      4979.64  5081.25   3157.5   7125.0   875.97
Jowar/Sorghum  2243.4